In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, make_scorer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,KFold, cross_val_score
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')


In [2]:
df = pd.read_csv('merged_dataset.csv')

print("\nDimensiones del DataFrame:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())


Dimensiones del DataFrame: (99441, 41)

Columnas:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivered', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'review_score', 'has_comment', 'avg_item_price', 'avg_freight_value', 'num_items', 'product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'total_payment', 'seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'estimated_delivery_days', 'actual_delivery_days', 'delivery_time_delta', 'is_late', 'total_order_value', 'freight_ratio', 'purchase_day_of_week', 'review_category']


In [3]:
# Seleccionar columnas relevantes
model_cols = [
    'delivered', 'num_items', 'avg_freight_value', 'total_order_value',
    'product_photos_qty', 'product_description_lenght',
    'estimated_delivery_days', 'actual_delivery_days', 'delivery_time_delta', 'is_late',
    'product_category_name_english', 'customer_state', 'seller_state', 'order_status',
    'review_category'
]

model_df = df[model_cols].copy()

In [4]:
# Preparar datos

class ToDenseTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.toarray() if hasattr(X, 'toarray') else X


# Agrupar categorías raras en product_category_name_english (<1% de las filas)
threshold = 0.01 * len(model_df)
category_counts = model_df['product_category_name_english'].value_counts()
rare_categories = category_counts[category_counts < threshold].index
model_df['product_category_name_english'] = model_df['product_category_name_english'].replace(rare_categories, 'Others')


sample_df = model_df.sample(n=20000, random_state=69)
X = sample_df.drop('review_category', axis=1)
y = sample_df['review_category']

# Verificar distribución de review_category
print("\nDistribución de review_category:")
print(y.value_counts(normalize=True) * 100)

print("\nDistribución de review_score:")
print(df['review_score'].value_counts().sort_index())


Distribución de review_category:
review_category
1    85.38
0    14.62
Name: proportion, dtype: float64

Distribución de review_score:
review_score
1.0    11344
2.0     3131
3.0     8120
4.0    19048
5.0    57798
Name: count, dtype: int64


#### Conjunto de datos para entrenamiento y prueba

In [5]:
X = df.drop(['review_score', 'review_category'], axis=1)
y_reg = df['review_score']  # Para regresión
y_clf = df['review_category']  # Para clasificación binaria

# Definir columnas numéricas y categóricas para el preprocesamiento
numerical_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]
categorical_cols = [col for col in X.columns if X[col].dtype == 'object']

print(f"Features numéricas: {len(numerical_cols)}")
print(f"Features categóricas: {len(categorical_cols)}")

Features numéricas: 22
Features categóricas: 17


In [6]:
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.3, random_state=42
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba: {X_test.shape[0]} muestras")

Conjunto de entrenamiento: 69608 muestras
Conjunto de prueba: 29833 muestras


VALIDACIÓN CRUZADA K-FOLD

In [ ]:
from sklearn.model_selection import KFold, cross_val_score


preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),  # Handle missing values for numerical columns
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),  # Handle missing values for categorical columns
            ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
        ]), categorical_cols)
    ], 
    remainder='passthrough'
)

# Función para mostrar resultados de validación cruzada
def print_cv_results(cv_scores, model_name, metric_name):
    print(f"{model_name} - {metric_name} con 5-fold CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print("\nClasificación Binaria - RandomForest con CV")

# Pipeline para RandomForest con preprocesamiento
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Validación cruzada con 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Validación cruzada para F1-Score
cv_f1_scores = cross_val_score(rf_pipeline, X, y_clf, cv=kf, scoring='f1')
print_cv_results(cv_f1_scores, "RandomForest", "F1-Score")

# Validación cruzada para Accuracy
cv_acc_scores = cross_val_score(rf_pipeline, X, y_clf, cv=kf, scoring='accuracy')
print_cv_results(cv_acc_scores, "RandomForest", "Accuracy")


Clasificación Binaria - RandomForest con CV
